# Fuse branches → submission.csv  (Track C integration)

Needs `trackc.py` + `b2_foundation.py` next to `data/`. Extracts the **baseline** (spatial, strong on d1) and **B2** (invariant, better on d2/d3) branches over all 6 pools, fuses them with **per-dataset RRF weights**, validates, and writes `submission.csv`.

Per-branch score CSVs are saved (`branch_baseline.csv`, `branch_b2.csv`) — teammates' B1/B3/B4 drop into the same `fuse_branches(...)` call. Extraction is ~10–25 min on GPU (754 volumes); progress prints as it goes.

In [1]:
import os, sys
FOLDER = '/shared-docker/Nicole'
os.chdir(FOLDER)
sys.path.insert(0, FOLDER)
print("now running from:", os.getcwd())
print("files:", [f for f in os.listdir('.') if f.endswith(('.py', '.ipynb'))])

now running from: /shared-docker/Nicole
files: ['Untitled.ipynb', 'run_pipeline_c.ipynb', 'b2_foundation.py', 'trackc.py', 'run_b2.ipynb', 'run_fuse_submit.ipynb']


In [2]:
import importlib, trackc, b2_foundation as b2
importlib.reload(trackc); importlib.reload(b2)
from trackc import *
import numpy as np, pandas as pd, torch

DATA_ROOT = '../data'; trackc.DATA_ROOT = DATA_ROOT
manifest = load_manifest(DATA_ROOT)
b2.download_weights()
model, device = b2.build_encoder()
print('ready on', device)

weights: model_swinvit.pt (411.2 MB)
SSL load: matched 94/126 encoder tensors (unmatched-from-ckpt: 40)
ready on cuda


## 1. Inline embedders (resample skipped for speed — we resize to a fixed grid anyway)

In [3]:
@torch.no_grad()
def b2_emb(path):
    vol = load_volume(path, resample_1mm=False, zscore=True, data_root=DATA_ROOT)
    vol = resize_to(vol, b2.INPUT_SIZE)
    x = torch.from_numpy(vol)[None, None].float().to(device)
    out = model.swinViT(x); feat = out[-1] if isinstance(out, (list, tuple)) else out
    v = torch.nn.functional.adaptive_avg_pool3d(feat, 1).flatten().cpu().numpy()
    return (v / (np.linalg.norm(v) + 1e-8)).astype(np.float32)

def base_emb(path):
    return embed_baseline(path, data_root=DATA_ROOT)

def embed_pool(df, id_col, img_col, fn):
    return {r[id_col]: fn(r[img_col]) for _, r in df.iterrows()}

## 2. Extract both branches over all 6 pools → branch score CSVs

In [4]:
base_rows, b2_rows = [], []
pools = [(k, v) for k, v in manifest.items() if k != 'train_pairs']
for (ds, split), pool in pools:
    for name, fn, bucket in [('baseline', base_emb, base_rows), ('b2', b2_emb, b2_rows)]:
        qE = embed_pool(pool.queries, 'query_id', 'query_image', fn)
        gE = embed_pool(pool.gallery, 'target_id', 'target_image', fn)
        df = embeddings_to_scores_df(qE, gE)
        bucket.append(df)
        print(f'{name:8s} {ds} {split}: {len(qE)}q x {len(gE)}g')

branch_baseline = pd.concat(base_rows, ignore_index=True)
branch_b2 = pd.concat(b2_rows, ignore_index=True)
branch_baseline.to_csv('branch_baseline.csv', index=False)
branch_b2.to_csv('branch_b2.csv', index=False)
print('saved branch_baseline.csv', branch_baseline.shape, '| branch_b2.csv', branch_b2.shape)

baseline dataset1 val: 40q x 40g
b2       dataset1 val: 40q x 40g
baseline dataset1 test: 100q x 100g
b2       dataset1 test: 100q x 100g
baseline dataset2 val: 40q x 40g
b2       dataset2 val: 40q x 40g
baseline dataset2 test: 100q x 100g
b2       dataset2 test: 100q x 100g
baseline dataset3 val: 20q x 20g
b2       dataset3 val: 20q x 20g
baseline dataset3 test: 77q x 77g
b2       dataset3 test: 77q x 77g
saved branch_baseline.csv (29529, 3) | branch_b2.csv (29529, 3)


In [5]:
def query_pool_map(manifest):
    m = {}
    for key, pool in manifest.items():
        if key == 'train_pairs': continue
        for q in pool.query_ids:
            m[q] = (pool.dataset, pool.split)
    return m

def fuse_branches(branch_dfs, manifest, weights_by_dataset=None, k=trackc.RRF_K):
    weights_by_dataset = weights_by_dataset or {}
    rankings = {name: scores_to_rankings(df) for name, df in branch_dfs.items()}
    pool_rankings = {}
    for key, pool in manifest.items():
        if key == 'train_pairs': continue
        ds, split = pool.dataset, pool.split
        w = weights_by_dataset.get(ds, {})
        branch_list, weight_list = [], []
        for name in branch_dfs:
            branch_list.append({q: rankings[name].get(q, []) for q in pool.query_ids})
            weight_list.append(float(w.get(name, 1.0)))
        pool_rankings[(ds, split)] = rrf(branch_list, weights=weight_list, k=k)
    return pool_rankings

print('fuse_branches ready')


fuse_branches ready


## 3. Per-dataset weighted fusion → validated submission.csv
d1: trust the spatial baseline; d2/d3: trust the invariant B2. (Teammates' branches will join this dict.)

In [6]:
weights = {
    'dataset1': {'baseline': 1.0, 'b2': 0.0},   # L1-optimal: baseline only
    'dataset2': {'baseline': 1.0, 'b2': 2.0},   # L2-optimal: B2-heavy fusion
    'dataset3': {'baseline': 1.0, 'b2': 2.0},   # d2 proxy (no d3 local label)
}

pool_rankings = fuse_branches({'baseline': branch_baseline, 'b2': branch_b2},
                              manifest, weights_by_dataset=weights)
sub = write_submission(pool_rankings, manifest, path='submission.csv')
validate_submission(sub, manifest)
print('wrote submission.csv'); sub.head(3)

submission OK: 377 rows, all rankings are valid same-pool permutations.
wrote submission.csv


,query_id,target_id_ranking
0,q_976703e4e366,g_db4a8dbe790b g_435a82857e20 g_da5aa13b1968 g...
1,q_fa1c256f9682,g_cdab0312f9ba g_018969561d96 g_9f1cf15a2d25 g...
2,q_511f87b5c69e,g_9f1cf15a2d25 g_0b251252e181 g_018969561d96 g...


## 4. Local sanity — fusion must not hurt d1 (held-out 50, d1 weights)

In [7]:
_, hold_df, gt = make_local_split(manifest['train_pairs'], n_holdout=50, seed=0)
qb = embed_pool(hold_df, 'query_id', 'query_image', base_emb)
gb = embed_pool(hold_df, 'target_id', 'target_image', base_emb)
q2 = embed_pool(hold_df, 'query_id', 'query_image', b2_emb)
g2 = embed_pool(hold_df, 'target_id', 'target_image', b2_emb)
r_base = rank_by_embeddings(qb, gb)
r_b2 = rank_by_embeddings(q2, g2)
r_fused = rrf([r_base, r_b2], weights=[1.0, 0.2])  # d1 weights
print('held-out d1 MRR  baseline =', round(mrr(r_base, gt), 4),
      '| b2 =', round(mrr(r_b2, gt), 4),
      '| fused(d1 wts) =', round(mrr(r_fused, gt), 4))

held-out d1 MRR  baseline = 0.7626 | b2 = 0.2546 | fused(d1 wts) = 0.6727


In [8]:
from scipy.ndimage import zoom
rng = np.random.default_rng(0)

def _base_d(path):
    v = resize_to(load_volume(path, resample_1mm=False, zscore=True, data_root=DATA_ROOT), (48,48,48))
    v = deform_volume(v, rng=rng); gx,gy,gz = np.gradient(v)
    v = np.sqrt(gx**2+gy**2+gz**2).ravel().astype('float32'); return v/(np.linalg.norm(v)+1e-8)

@torch.no_grad()
def _b2_d(path):
    v = resize_to(load_volume(path, resample_1mm=False, zscore=True, data_root=DATA_ROOT), b2.INPUT_SIZE)
    v = deform_volume(v, rng=rng); x = torch.from_numpy(v)[None,None].float().to(device)
    out = model.swinViT(x); f = out[-1] if isinstance(out,(list,tuple)) else out
    v = torch.nn.functional.adaptive_avg_pool3d(f,1).flatten().cpu().numpy(); return v/(np.linalg.norm(v)+1e-8)

gb_l2 = embed_pool(hold_df, 'target_id', 'target_image', _base_d)
g2_l2 = embed_pool(hold_df, 'target_id', 'target_image', _b2_d)

rb1, rq1 = rank_by_embeddings(qb,gb), rank_by_embeddings(q2,g2)
rb2, rq2 = rank_by_embeddings(qb,gb_l2), rank_by_embeddings(q2,g2_l2)
print(' w_b2 |  L1(d1)  L2(d2/d3 proxy)')
for w in [0.0, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0]:
    m1 = mrr(rrf([rb1, rq1], weights=[1.0, w]), gt)
    m2 = mrr(rrf([rb2, rq2], weights=[1.0, w]), gt)
    print(f'{w:4.1f}  |  {m1:.3f}   {m2:.3f}')


 w_b2 |  L1(d1)  L2(d2/d3 proxy)
 0.0  |  0.763   0.098
 0.1  |  0.717   0.102
 0.2  |  0.673   0.103
 0.5  |  0.544   0.107
 1.0  |  0.454   0.116
 2.0  |  0.355   0.121
 5.0  |  0.293   0.111


## Done
`submission.csv` is validated (377 rows) — ready to upload to Kaggle for the **first calibration submission** (locks local↔leaderboard agreement).

Paste back: the **`held-out d1 MRR`** line and the **`submission OK`** line. When a teammate hands you a `branch_*.csv` (same `[query_id,target_id,score]` format), add it to the `fuse_branches({...})` dict and re-run cell 3 — no other change.